In [0]:
 folder_path = "abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Silver/flight_final_delta"
df = spark.read \
    .format("delta") \
    .load(folder_path)

display(df)

In [0]:
from pyspark.sql import functions as F

df = df.withColumns({
    "origin_lat": F.round(F.col("origin_lat"), 2),
    "origin_lon": F.round(F.col("origin_lon"), 2),
    "dest_lat": F.round(F.col("dest_lat"), 2),
    "dest_lon": F.round(F.col("dest_lon"), 2)
})

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Read Silver flight data
df = spark.read \
    .format("delta") \
    .load("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Silver/flight_final_delta")

# 2. Select required columns
df_monthly = df.select(
    "Year",
    "Month",
    "Reporting_Airline",
    F.col("Cancelled").cast("double").alias("Cancelled"),
    F.col("Diverted").cast("double").alias("Diverted"),
    F.col("ArrDel15").cast("double").alias("ArrDel15"),
    F.col("ArrDelayMinutes").cast("double").alias("ArrDelayMinutes"),
    F.col("CarrierDelay").cast("double").alias("CarrierDelay"),
    F.col("WeatherDelay").cast("double").alias("WeatherDelay"),
    F.col("SecurityDelay").cast("double").alias("SecurityDelay"),
    F.col("Distance").cast("double").alias("Distance")
)

# 3. Monthly airline analytics
flightdb_monthly_df = (
    df_monthly
    .groupBy(
        "Year",
        "Month",
        "Reporting_Airline"
    )
    .agg(
        F.count("*").alias("total_flights"),

        F.sum(
            F.when(
                (F.col("Cancelled") == 0) &
                (F.col("Diverted") == 0),
                1
            ).otherwise(0)
        ).alias("flown_flights"),

        F.sum(
            F.when(F.col("Cancelled") == 1, 1)
             .otherwise(0)
        ).alias("cancelled_flights"),

        F.sum(
            F.when(F.col("Diverted") == 1, 1)
             .otherwise(0)
        ).alias("diverted_flights"),

        F.sum(
            F.when(F.col("ArrDel15") == 1, 1)
             .otherwise(0)
        ).alias("delayed_flights"),

        F.round(
            F.avg("ArrDelayMinutes"), 2
        ).alias("avg_delay"),

        F.round(
            F.avg(
                F.when(
                    (F.col("Cancelled") == 0) &
                    (F.col("Diverted") == 0),
                    F.when(F.col("ArrDel15") == 0, 1).otherwise(0)
                )
            ) * 100,
            2
        ).alias("on_time_percentage"),

        F.round(
            F.avg(
                F.when(F.col("Cancelled") == 1, 1)
                 .otherwise(0)
            ) * 100,
            2
        ).alias("cancellation_rate_percentage"),

        F.round(
            F.avg("CarrierDelay"), 2
        ).alias("avg_carrier_delay"),

        F.round(
            F.avg("WeatherDelay"), 2
        ).alias("avg_weather_delay"),

        F.round(
            F.avg("SecurityDelay"), 2
        ).alias("avg_security_delay"),

        F.round(
            F.avg("Distance"), 2
        ).alias("avg_distance"),

        F.round(
            F.sum("Distance"), 2
        ).alias("total_distance")
    )
)

# 4. Rank airlines within each month
window_spec = Window.partitionBy(
    "Year",
    "Month"
).orderBy(
    F.col("total_flights").desc()
)

flightdb_monthly_df = flightdb_monthly_df.withColumn(
    "monthly_flight_ranking",
    F.row_number().over(window_spec)
)

# 5. Display Gold result
display(flightdb_monthly_df)

In [0]:
df_route = df.select(
    "Year",
    "Origin",
    "OriginCityName",
    "OriginStateName",
    "Dest",
    "DestCityName",
    "DestStateName",
    F.col("Cancelled").cast("double").alias("Cancelled"),
    F.col("Diverted").cast("double").alias("Diverted"),
    F.col("ArrDel15").cast("double").alias("ArrDel15"),
    F.col("ArrDelayMinutes").cast("double").alias("ArrDelayMinutes"),
    F.col("CarrierDelay").cast("double").alias("CarrierDelay"),
    F.col("WeatherDelay").cast("double").alias("WeatherDelay"),
    F.col("SecurityDelay").cast("double").alias("SecurityDelay"),
    F.col("Distance").cast("double").alias("Distance")
)

route_analytics_df = (
    df_route
    .groupBy(
        "Year",
        "Origin",
        "OriginCityName",
        "OriginStateName",
        "Dest",
        "DestCityName",
        "DestStateName"
    )
    .agg(
        F.count("*").alias("total_flights"),

        F.sum(
            F.when(
                (F.col("Cancelled") == 0) &
                (F.col("Diverted") == 0),
                1
            ).otherwise(0)
        ).alias("served_flights"),

        F.sum(
            F.when(F.col("Cancelled") == 1, 1)
             .otherwise(0)
        ).alias("cancelled_flights"),

        F.sum(
            F.when(F.col("Diverted") == 1, 1)
             .otherwise(0)
        ).alias("diverted_flights"),

        F.sum(
            F.when(F.col("ArrDel15") == 1, 1)
             .otherwise(0)
        ).alias("delayed_flights"),

        F.round(
            F.avg("ArrDelayMinutes"), 2
        ).alias("avg_delay"),

        F.round(
            F.avg(
                F.when(
                    (F.col("Cancelled") == 0) &
                    (F.col("Diverted") == 0),
                    F.when(F.col("ArrDel15") == 0, 1).otherwise(0)
                )
            ) * 100,
            2
        ).alias("on_time_percentage"),

        F.round(
            F.avg(
                F.when(F.col("Cancelled") == 1, 1)
                .otherwise(0)
            ) * 100,
            2
        ).alias("cancellation_rate_percentage"),

        F.round(F.avg("CarrierDelay"), 2)
         .alias("avg_carrier_delay"),

        F.round(F.avg("WeatherDelay"), 2)
         .alias("avg_weather_delay"),

        F.round(F.avg("SecurityDelay"), 2)
         .alias("avg_security_delay"),

        F.round(F.avg("Distance"), 2)
         .alias("avg_distance_km"),

        F.round(F.sum("Distance"), 2)
         .alias("total_distance_km")
    )
    .select(
        F.col("Year"),
        F.col("Origin").alias("origin_airport_code"),
        F.col("OriginCityName").alias("origin_city"),
        F.col("OriginStateName").alias("origin_state"),
        F.col("Dest").alias("dest_airport_code"),
        F.col("DestCityName").alias("dest_city"),
        F.col("DestStateName").alias("dest_state"),
        "total_flights",
        "served_flights",
        "cancelled_flights",
        "diverted_flights",
        "delayed_flights",
        "avg_delay",
        "on_time_percentage",
        "cancellation_rate_percentage",
        "avg_carrier_delay",
        "avg_weather_delay",
        "avg_security_delay",
        "avg_distance_km",
        "total_distance_km"
    )
)

display(route_analytics_df)


In [0]:
df_airport = df.select(
    "Year",
    "Month",
    "Origin",
    "OriginCityName",
    "OriginStateName",
    "origin_lat",
    "origin_lon",
    F.col("Cancelled").cast("double").alias("Cancelled"),
    F.col("DepDelayMinutes").cast("double").alias("DepDelayMinutes"),
    F.col("DepDel15").cast("double").alias("DepDel15"),
    F.col("AirTime").cast("double").alias("AirTime"),
    F.col("Distance").cast("double").alias("Distance"),
    "Reporting_Airline",
    "Dest"
)

airport_monthly_df = (
    df_airport
    .groupBy(
        "Year",
        "Month",
        "Origin",
        "OriginCityName",
        "OriginStateName",
        "origin_lat",
        "origin_lon"
    )
    .agg(
        F.count("*").alias("departures"),

        F.sum(
            F.when(F.col("Cancelled") == 1, 1)
             .otherwise(0)
        ).alias("cancelled_departures"),

        F.round(
            F.avg("DepDelayMinutes"), 2
        ).alias("avg_dep_delay"),

        F.round(
            F.avg(
                F.when(
                    F.col("Cancelled") == 0,
                    F.when(F.col("DepDel15") == 0, 1).otherwise(0)
                )
            ) * 100,
            2
        ).alias("dep_on_time_pct"),

        F.round(
            F.avg("AirTime"), 2
        ).alias("avg_airtime"),

        F.round(
            F.avg("Distance"), 2
        ).alias("avg_route_distance"),

        F.countDistinct(
            "Reporting_Airline"
        ).alias("airlines_operating"),

        F.countDistinct(
            "Dest"
        ).alias("unique_destinations")
    )
    .select(
        F.col("Year").alias("flight_year"),
        F.col("Month").alias("flight_month"),

        F.format_string(
            "%04d-%02d",
            F.col("Year").cast("int"),
            F.col("Month").cast("int")
        ).alias("year_month"),

        F.col("Origin").alias("origin_code"),

        F.lit(None)
         .cast("string")
         .alias("origin_airport_name"),

        F.col("OriginCityName").alias("origin_city"),
        F.col("OriginStateName").alias("origin_state"),

        "origin_lat",
        "origin_lon",

        "departures",
        "cancelled_departures",
        "avg_dep_delay",
        "dep_on_time_pct",
        "avg_airtime",
        "avg_route_distance",
        "airlines_operating",
        "unique_destinations",

        F.current_timestamp().alias("gold_processed")
    )
)

display(airport_monthly_df)

In [0]:

from pyspark.sql import functions as F

# 1. Read Silver flight data
df = spark.read.format("delta").load("abfss://bigdata-5@bigdatastoragee.dfs.core.windows.net/Silver/flight_final_delta")

# 2. Select the required columns from Silver
df_delay = df.select(
    "Year",
    "Month",
    "Reporting_Airline",
    F.col("ArrDel15").cast("double").alias("ArrDel15"),
    F.col("DepDelayMinutes").cast("double").alias("DepDelayMinutes"),
    F.col("CarrierDelay").cast("double").alias("CarrierDelay"),
    F.col("WeatherDelay").cast("double").alias("WeatherDelay"),
    F.col("NASDelay").cast("double").alias("NASDelay"),
    F.col("SecurityDelay").cast("double").alias("SecurityDelay"),
    F.col("LateAircraftDelay").cast("double").alias("LateAircraftDelay")
)

# 2. Create Delay Cause Analysis DataFrame
delay_cause_df = (
    df_delay
    .groupBy(
        "Year",
        "Month",
        "Reporting_Airline"
    )
    .agg(
        # Total flights delayed by 15 minutes or more
        F.sum(
            F.when(F.col("ArrDel15") == 1, 1).otherwise(0)
        ).alias("total_delayed_flights"),

        # Total delay minutes
        F.sum("DepDelayMinutes").alias("total_delay_minutes"),

        # Delay minutes by cause
        F.sum("CarrierDelay").alias("carrier_delay_minutes"),
        F.sum("WeatherDelay").alias("weather_delay_minutes"),
        F.sum("NASDelay").alias("nas_delay_minutes"),
        F.sum("SecurityDelay").alias("security_delay_minutes"),
        F.sum("LateAircraftDelay").alias("late_aircraft_delay_minutes"),

        # Percentage of flights affected by each delay cause
        F.round(
            F.avg(
                F.when(F.col("CarrierDelay") > 0, 1.0)
                 .otherwise(0.0)
            ) * 100, 2
        ).alias("carrier_delay_percentage"),

        F.round(
            F.avg(
                F.when(F.col("WeatherDelay") > 0, 1.0)
                 .otherwise(0.0)
            ) * 100, 2
        ).alias("weather_delay_percentage"),

        F.round(
            F.avg(
                F.when(F.col("NASDelay") > 0, 1.0)
                 .otherwise(0.0)
            ) * 100, 2
        ).alias("nas_delay_percentage"),

        F.round(
            F.avg(
                F.when(F.col("SecurityDelay") > 0, 1.0)
                 .otherwise(0.0)
            ) * 100, 2
        ).alias("security_delay_percentage"),

        F.round(
            F.avg(
                F.when(F.col("LateAircraftDelay") > 0, 1.0)
                 .otherwise(0.0)
            ) * 100, 2
        ).alias("late_aircraft_delay_percentage")
    )
    .select(
        F.col("Year").alias("flight_year"),
        F.col("Month").alias("flight_month"),
        F.col("Reporting_Airline").alias("airline_code"),
        "total_delayed_flights",
        "total_delay_minutes",
        "carrier_delay_minutes",
        "weather_delay_minutes",
        "nas_delay_minutes",
        "security_delay_minutes",
        "late_aircraft_delay_minutes",
        "carrier_delay_percentage",
        "weather_delay_percentage",
        "nas_delay_percentage",
        "security_delay_percentage",
        "late_aircraft_delay_percentage",
        F.current_timestamp().alias("gold_processed_ts")
    )
)

# 3. Display the Gold DataFrame
display(delay_cause_df)


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.secrets.create_scope(scope="snowflake-secrets")
w.secrets.put_secret(
    scope="snowflake-secrets",
    key="snowflake-password",
    string_value="YOUR_SNOWFLAKE_PASSWORD"
)

In [0]:
dbutils.widgets.text("sf_password", "")

In [0]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

conn = snowflake.connector.connect(
    account="ORWLDEH-PH02080",
    user="VEDANG34343434222",
    password=dbutils.widgets.get("sf_password"),
    database="DATABRICK_PROJECT_DB",
    schema="GOLD_SCHEMA",
    warehouse="COMPUTE_WH",
    role="ACCOUNTADMIN"
)

# Create database and schema if they don't exist, then set context
cur = conn.cursor()
cur.execute("CREATE DATABASE IF NOT EXISTS DATABRICK_PROJECT_DB")
cur.execute("USE DATABASE DATABRICK_PROJECT_DB")
cur.execute("CREATE SCHEMA IF NOT EXISTS GOLD_SCHEMA")
cur.execute("USE SCHEMA GOLD_SCHEMA")
cur.close()

tables = {
    "FLIGHT_SUMMARY_TABLE": flightdb_monthly_df,
    "ROUTE_BASED_INSIGHTS": route_analytics_df,
    "DELAY_CAUSE_ANALYSIS": delay_cause_df,
    "AIRPORT_MONTHLY_PERFORMANCE": airport_monthly_df,
}

import pandas as pd

for table_name, spark_df in tables.items():
    # Reconstruct a clean pandas DF to strip Spark Connect PlanMetrics metadata
    raw = spark_df.toPandas()
    pdf = pd.DataFrame(raw.to_dict(orient="list"))
    pdf.columns = [c.upper() for c in pdf.columns]
    write_pandas(conn, pdf, table_name, auto_create_table=True,
                 overwrite=True, use_logical_type=True)
    print(f"{table_name} saved ({len(pdf)} rows)")

conn.close()
print("All 4 DataFrames saved to Snowflake!")